In [12]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
import pickle
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [13]:
df = pd.read_csv('dataset.csv', encoding="ISO-8859-1", nrows=20000)


In [14]:
nltk.download('stopwords')
ps = PorterStemmer()
data = []

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/esheta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
for i in range(len(df)):
    review = df['SentimentText'][i]
    review = re.sub('[^a-zA-Z]', ' ', review)
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if word not in set(stopwords.words('english'))]
    review = ' '.join(review)
    data.append(review)

In [16]:
cv = CountVectorizer(max_features=30000)
X = cv.fit_transform(data).toarray()

In [17]:
y = df.iloc[:, 1].values

In [18]:
pickle.dump(cv, open("cv.pkl", "wb"))

In [19]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # CPU only
model = Sequential()
model.add(Dense(units=64, kernel_initializer='uniform', activation='relu', input_dim=X.shape[1]))
model.add(Dense(units=16, kernel_initializer='uniform', activation='relu'))
model.add(Dense(units=1, kernel_initializer='uniform', activation='sigmoid'))



/opt/anaconda3/envs/tf/lib/python3.10/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [21]:
model.fit(X, y, batch_size=16, epochs=5)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.7251 - loss: 0.5525
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8338 - loss: 0.3953
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8729 - loss: 0.3229
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8836 - loss: 0.2970
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8920 - loss: 0.2848


In [22]:
model.save("model.h5")